# KV Caching with Valkey

**Difficulty:** Intermediate · Python · ~20 min

Caching the attention KV an LLM computes for a shared prompt prefix cuts cost
per request and shortens time-to-first-token. This walkthrough stores that KV
in Valkey and demonstrates how it can be reused across vLLM instances.

When an LLM answers a prompt, it first computes attention key/value tensors for
every input token. This is the prefill step: its cost is proportional to the
prompt length, and the result is normally thrown away after each request.
Whenever a later request shares a prefix with an earlier one, prefill recomputes
the same KV from scratch.

Caching that KV and reloading it skips the recompute: you pay for prefill once
and reuse it. Shared prefixes are common in practice:

- **Multi-turn chat:** each turn typically resends much of the conversation so
  far, so a later turn shares a long prefix with the earlier ones. How much is
  resent depends on the application, but something almost always is.
- **A common preamble:** a system prompt, tool and function definitions, MCP
  context, skills, or other instructions sit in front of many requests.
- **RAG and agentic memory:** retrieved passages or recalled memories are often
  reused across requests, depending on the retrieval strategy.

[LMCache](https://github.com/LMCache/LMCache) is the KV-cache layer that plugs
into vLLM and does this. It can store KV in several places, including local
memory, disk, or a remote backend; this walkthrough runs a standalone LMCache
server that uses Valkey as its backend. vLLM's own built-in prefix cache is limited to what one engine can hold
and is not shared with other engines. Valkey is a good fit when several vLLM
instances must read one cache, and that cache needs to be larger than any single
host can hold. The instances sharing a cache must run the same model and tensor
configuration, since the cached KV is specific to both.

## Prerequisites

- **A container runtime with Compose** (Docker Compose or compatible). The
  stack is four containers: Valkey, a standalone LMCache server, and two vLLM
  replicas.
- **Memory for the stack.** Give your container runtime **at least 10 GiB** of
  RAM. Each vLLM replica needs roughly 3–4 GiB; the LMCache server and Valkey
  add a little more. On a smaller machine, run one replica and skip the
  two-replica step near the end.
- **Disk space.** The stack pulls about **5 GiB** of images (the vLLM CPU
  image alone is ~3.7 GiB) plus the model weights on first run. Make sure
  the container runtime's disk has room.
- **Python 3.12** for the notebook client, matching the notebook's kernel. (LMCache currently supports 3.10–3.13; 3.12 is what this cookbook is pinned and tested against.)

**The stack:**

| Component | Role |
| --- | --- |
| [vLLM](https://docs.vllm.ai) (CPU image) | Serves the model; hands KV to the LMCache server via the multi-process connector |
| [LMCache](https://docs.lmcache.ai) server | Owns KV storage: an L1 tier in its own memory, writing through to Valkey (L2) |
| [Valkey](https://valkey.io) | The shared KV-cache store (LMCache's L2 backend) |
| Model (`facebook/opt-125m`) | Small and ungated, so no GPU or token needed |

Install this notebook's client dependencies (from this directory):

```bash
python3.12 -m venv .venv
.venv/bin/python -m pip install -r requirements.txt
.venv/bin/python -m ipykernel install --user --name kv-caching-valkey
```

Then select the `kv-caching-valkey` kernel when you open this notebook.

> **CPU vs GPU:** this walkthrough runs entirely on CPU so anyone can follow it.
> A GPU speeds up the model's own compute. It does not change the caching
> behavior.

## How to run this notebook

This is a runnable notebook. There are two parts:

1. **One-time terminal setup (the `bash` blocks below).** The commands in the
   fenced `bash` blocks (`docker compose up`, creating the virtualenv,
   installing the kernel) run once in a terminal, not in the notebook. They
   start the stack and set up the kernel this notebook runs on.
2. **Run the code cells top to bottom.** Once the stack is up and you've
   opened this notebook on the `kv-caching-valkey` kernel, run each code cell
   in order (Shift+Enter, or Run All). Each cell prints what to look for, such
   as key counts and Valkey `GET` calls, so you can watch the cache fill up and
   get reused as you go. Run them in order, because later cells build on the
   conversation state that earlier cells set up.

## Setup

Start the whole stack, Valkey, the LMCache server, and both vLLM replicas, with
one command from this directory:

```bash
docker compose up -d --wait
```

`--wait` blocks until every container reports healthy. **The first run is slow:**
the vLLM image is large, each replica loads the model on startup, and the
LMCache server installs its dependencies, so give it several minutes. Subsequent
runs reuse the cached image and model weights.

### Pointing LMCache at Valkey

On CPU, vLLM has no in-worker KV connector, so each replica uses the
multi-process connector (`LMCacheMPConnector`) and hands its KV to a standalone
`lmcache server`. That server is what talks to Valkey, through its L2 adapter.
In [`docker-compose.yml`](docker-compose.yml) the server starts with:

```bash
lmcache server --l1-size-gb 2 --eviction-policy noop \
  --l2-store-policy skip_l1 --supported-transfer-mode auto \
  --l2-adapter '{"type":"valkey","startup_nodes":"valkey:6379"}'
```

That `--l2-adapter` spec is what makes Valkey the backing store.
`--l2-store-policy skip_l1` keeps the server's L1 tier as a write buffer only,
so a cache *read* comes from Valkey rather than local memory — that is what
makes Valkey's role visible as `GET` calls later in this notebook. A production
deployment usually keeps L1 hot for lower-latency reads and lets Valkey back it
as the larger shared tier. [`lmcache_config.yaml`](lmcache_config.yaml)
documents the same spec and the options you are most likely to change. See the
[LMCache MP server configuration reference](https://docs.lmcache.ai/mp/configuration.html)
for the full option set.

> **Security:** This stack uses no authentication or TLS for simplicity.
> For any non-localhost deployment, enable authentication and TLS.
> See the [Valkey security documentation](https://valkey.io/topics/security/).

In [ ]:
# ── First cell to run (after the stack is up). Run the code cells top to
# ── bottom from here; each one prints what to look for.
# A tiny bit of shared setup: talk to each replica over its OpenAI-compatible
# HTTP API, and read the KV-cache state straight from Valkey via the CLI.
import subprocess
from openai import OpenAI

MODEL = "facebook/opt-125m"

# The two vLLM replicas, each mapped to a host port by docker-compose.yml.
replica_a = OpenAI(base_url="http://localhost:8001/v1", api_key="not-needed")
replica_b = OpenAI(base_url="http://localhost:8002/v1", api_key="not-needed")


def valkey_cli(*args):
    """Run a valkey-cli command inside the Valkey container and return stdout."""
    out = subprocess.run(
        ["docker", "exec", "valkey", "valkey-cli", *args],
        capture_output=True, text=True,
    )
    return out.stdout.strip()


def dbsize():
    """Number of keys in Valkey. Each cached KV chunk is stored as keys here."""
    return int(valkey_cli("DBSIZE"))


def kv_keys():
    """The KV-chunk keys LMCache has written to Valkey (its L2 tier)."""
    out = valkey_cli("KEYS", "*")
    return [k for k in out.splitlines() if k]


def valkey_get_calls():
    """Cumulative Valkey GET/MGET calls (from INFO commandstats).

    The LMCache server runs with --l2-store-policy skip_l1, so L1 is only a
    write buffer and a cache *read* has to come from Valkey. Reuse therefore
    shows up here as GET calls.
    """
    total = 0
    for line in valkey_cli("INFO", "commandstats").splitlines():
        if line.startswith(("cmdstat_get:", "cmdstat_mget:")):
            for field in line.split(":", 1)[1].split(","):
                if field.startswith("calls="):
                    total += int(field.split("=", 1)[1])
    return total


print("Setup ready.")

Confirm the whole stack is live before going further: Valkey answers `PING`, and
both replicas list the model on `/v1/models`.

In [ ]:
print("valkey ping :", valkey_cli("PING"))
print("replica A   :", [m.id for m in replica_a.models.list().data])
print("replica B   :", [m.id for m in replica_b.models.list().data])

## Example: a multi-turn chat

We walk through one scenario start to finish, so you can map it onto your own
use. We hold a short multi-turn conversation with replica A and watch the KV
cache in Valkey between turns. Then we send the same conversation to replica B
to prove the cache is actually shared.

We keep the conversation in an explicit `messages` list and grow it turn by
turn. That is what makes each turn share the previous turn's prefix, the same
thing that happens in any real chat UI.

In [ ]:
SYSTEM_PROMPT = (
    "You are a concise assistant that answers questions about world geography. "
    "Keep every answer to a single short sentence."
)

# The conversation so far. We append to this between turns.
messages = [{"role": "system", "content": SYSTEM_PROMPT}]


def ask(client, question, conversation):
    """Send the whole conversation + a new question, return the answer text."""
    conversation.append({"role": "user", "content": question})
    resp = client.chat.completions.create(
        model=MODEL, messages=conversation, max_tokens=32, temperature=0.0,
    )
    answer = resp.choices[0].message.content
    conversation.append({"role": "assistant", "content": answer})
    return answer

### Turn 1: cold

Nothing is cached yet. Replica A has to prefill the full prompt (system prompt +
first question), computing KV for every token from scratch. What gets cached is
exactly that KV: the attention key/value tensors for the prompt's tokens, split
into fixed-size chunks. As replica A prefills, LMCache hands those KV chunks to
the LMCache server, which writes them to Valkey.

We record Valkey's key count before and after. A cold turn should make it
**grow**: the freshly computed KV blocks have landed in Valkey.

In [ ]:
before = dbsize()
answer1 = ask(replica_a, "What is the capital of France?", messages)
after = dbsize()

print("Q1 answer     :", answer1)
print("Valkey dbsize :", before, "->", after)
print("KV blocks stored in Valkey:", after - before)

### Turn 2: warm, shared prefix

Now the follow-up. The conversation we send is `system + Q1 + A1 + Q2`, so its
first three parts are byte-for-byte the prefix replica A just prefilled. Only
the new question at the end is new.

**How the hit works.** LMCache splits the token sequence into fixed-size chunks
(`chunk_size: 256`) and keys each chunk by a hash of the tokens in that chunk
and every chunk before it. The Valkey key is a pure function of
`(model, world_size, worker_id, chunk_hash, dtype)`
(`vllm@<model>@<world>@<worker>@<hash>` on the wire), so two requests that start
with the same tokens produce the identical key. A write by one is a hit for the
other. The match is on an exact token prefix: identical tokens from the
start, up to the point where the two requests diverge. Turn 2 shares every token
of the system prompt and first exchange, so those chunks are a hit. LMCache
loads their KV from Valkey and vLLM only prefills the handful of new tokens in
Q2.

Two signals here. The cached prefix is **not stored again**: `dbsize` does not
grow for the shared prefix (only the handful of new tokens in Q2 can add
anything). And the prefix's KV is **read back from Valkey**: because the server
runs with `--l2-store-policy skip_l1` (L1 is a write buffer only), the read
comes from Valkey as GET calls rather than from an in-memory tier. So LMCache
loads the prefix's KV from Valkey instead of vLLM recomputing it.

In [ ]:
gets_before = valkey_get_calls()
before = dbsize()
answer2 = ask(replica_a, "What about Italy?", messages)
after = dbsize()
gets_after = valkey_get_calls()

print("Q2 answer            :", answer2)
print("Valkey dbsize        :", before, "->", after)
print("new KV blocks stored :", after - before)
print("Valkey GET calls      :", gets_before, "->", gets_after)
print()
print("The shared prefix (system + Q1 + A1) was already cached, so it is a hit,")
print("not a re-store: dbsize barely moves. The GET calls are LMCache loading")
print("that prefix's KV back from Valkey instead of vLLM recomputing it.")


### A second replica hits the first replica's cache

Replica B has served no traffic and computed no KV of its own. We send it the
same conversation prefix replica A built up. The KV for that prefix lives in
Valkey, not inside replica A's process, so replica B finds it there and loads
it instead of prefilling from scratch.

The signal: replaying replica A's prefix on replica B **reads that KV back from
Valkey** (GET calls increment) and stores **no new keys** (`dbsize` is
unchanged), because the prefix's KV is already in the shared store. Replica B
reuses what replica A computed. An in-process cache on replica A would be
invisible to replica B; putting the cache in Valkey is what makes this
cross-replica reuse possible.

In [ ]:
# Replay replica A's exact conversation prefix (everything up to, but not
# including, replica A's last answer) as a fresh request to replica B.
shared_prefix = messages[:-1]

gets_before = valkey_get_calls()
before = dbsize()
resp = replica_b.chat.completions.create(
    model=MODEL, messages=shared_prefix, max_tokens=32, temperature=0.0,
)
after = dbsize()
gets_after = valkey_get_calls()

print("Replica B answer     :", resp.choices[0].message.content)
print("Valkey dbsize        :", before, "->", after, "(unchanged == not re-stored)")
print("Valkey GET calls      :", gets_before, "->", gets_after, "(went up == read from Valkey)")
print()
print("Replica B served no traffic of its own, yet answered by reading the KV")
print("replica A wrote to Valkey — the cache is genuinely shared through Valkey.")


### What it saves: cold vs. warm latency

The cache-hit signals above show *that* KV is reused. Here is *why it's worth
it*: time the same long prefix twice. The first request is cold — vLLM prefills
every token. The second shares the whole prefix, so LMCache loads that KV from
Valkey and vLLM skips the prefill.

On CPU with a tiny model like `opt-125m` the absolute numbers are small and a
little noisy, so treat the **direction**, not the exact milliseconds, as the
point. The saving grows with prompt length and model size — on a GPU serving a
production-sized model over a long shared prefix, skipping prefill is the
difference this cache is built for.

In [ ]:
import time

# A fresh, long prefix unique to this run, so the first request is genuinely
# cold (nothing cached yet).
timing_prefix = [
    {"role": "system", "content": SYSTEM_PROMPT + " " + "Reference context. " * 80},
    {"role": "user", "content": "What is the capital of France?"},
]


def timed_request(client, conversation):
    start = time.perf_counter()
    client.chat.completions.create(
        model=MODEL, messages=conversation, max_tokens=32, temperature=0.0,
    )
    return time.perf_counter() - start


cold_s = timed_request(replica_a, timing_prefix)   # prefill + store to Valkey
warm_s = timed_request(replica_a, timing_prefix)   # load KV from Valkey, skip prefill

print(f"cold (prefill)      : {cold_s * 1000:7.1f} ms")
print(f"warm (cache hit)    : {warm_s * 1000:7.1f} ms")
if warm_s < cold_s:
    print(f"saved               : {(cold_s - warm_s) * 1000:7.1f} ms "
          f"({(1 - warm_s / cold_s) * 100:.0f}% faster)")
else:
    print("(no speedup this run — expected on CPU with a tiny model; see the note above)")


## What else is in this directory

The notebook drives the demo, but the runnable stack is defined by the files
alongside it:

| File | Purpose |
| --- | --- |
| [`docker-compose.yml`](docker-compose.yml) | Starts Valkey, the LMCache server, and two vLLM replicas (`vllm-a`, `vllm-b`) sharing one Valkey |
| [`lmcache_config.yaml`](lmcache_config.yaml) | Documents the Valkey L2 adapter spec — the line that makes Valkey the LMCache backend |
| [`chat_template.jinja`](chat_template.jinja) | Minimal chat template so `opt-125m` can serve `/v1/chat/completions` |
| [`requirements.txt`](requirements.txt) | Pinned Python dependencies for this notebook's client |
| [`test_lmcache_valkey.py`](test_lmcache_valkey.py) | Tests: config + compose wiring always; live cache-hit signals when the stack is up |

### Running the tests

The config and compose-wiring tests run without the stack. The live tests
exercise the cache-hit signals and run only when the stack is up:

```bash
docker compose up -d --wait
.venv/bin/python -m pytest test_lmcache_valkey.py -v
```

## Cleanup

Tear down all four containers, Valkey, the LMCache server, and both vLLM replicas:

```bash
docker compose down
```

**What persists:** the KV cache lives in Valkey (the LMCache server's L2 tier),
so it survives an individual vLLM replica or the LMCache server restarting. The cache is lost when
`docker compose down` removes the container, unless you mount a data volume and
enable append-only persistence to keep it across container recreation. The
`hf-cache` volume that holds the downloaded model weights does persist across
`down`/`up`, so later runs skip the re-download.

## What you learned

- **Why Valkey:** it holds the KV cache *outside* any single vLLM process, so
  one cache can be shared across many vLLM instances and can outlive any single
  one of them. Replica B reused a prefix that replica A had computed, something
  an in-process cache cannot do. (The cache is in-memory; it only survives a
  Valkey restart if you enable persistence.)
- **The multi-turn payoff:** each turn resends the prior turn's prefix, so the
  warm turn loaded that prefix's KV from Valkey instead of recomputing it.
- **The mechanism:** LMCache keys each KV chunk by a hash of its token prefix,
  so requests sharing an exact token prefix land on the same Valkey keys. A
  cold turn stores them, and later turns and other replicas read them back from
  Valkey. (This demo sets `--l2-store-policy skip_l1` so reads come straight
  from Valkey and its role is visible; a production deployment usually keeps the
  L1 tier hot for lower-latency reads.)

## Next steps

- **Scale out.** The natural continuation of the two-replica step: run *more*
  vLLM replicas, all sharing this one Valkey. Every replica warms the same cache
  and reads every other replica's prefixes, so the hit rate climbs as you add
  replicas. (This is scaling the vLLM fleet against one Valkey, not adding Valkey
  replicas.)
- **Take it to production.** See the
  [LMCache production deployment guide](https://docs.lmcache.ai/production/docker_deployment.html)
  for running LMCache under a served vLLM, and the
  [LMCache Valkey/Redis backend docs](https://docs.lmcache.ai/kv_cache/redis.html)
  for cluster mode, high availability, and tuning the remote store.